<a href="https://colab.research.google.com/github/kinshukpcs24-ux/BIS_Lab/blob/main/1BM24CS137_ACO_PCB_Routing/1BM24CS137_ACO_PCB_Routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import random

class AntColonyPCB:
    def __init__(self, size=10, nets=None, obstacles=None, num_ants=10, generations=50, alpha=1.0, beta=2.0, evaporation=0.1, Q=1.0):
        """
        size: Dimension of the n x n board
        nets: List of tuples representing traces to route: [(start_coord, end_coord), ...]
        obstacles: Set of coordinates representing prohibited regions
        """
        self.size = size
        self.nets = nets if nets is not None else []
        self.obstacles = set(obstacles) if obstacles is not None else set()
        self.num_ants = num_ants
        self.generations = generations
        self.alpha = alpha
        self.beta = beta
        self.evaporation = evaporation
        self.Q = Q

        self.num_nets = len(self.nets)
        # Individual pheromone tables for each net on this single-layer grid
        self.pheromones = np.ones((self.num_nets, self.size, self.size, 4)) * 0.1

    def get_neighbors(self, pos):
        r, c = pos
        neighbors = []
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up (0), Down (1), Left (2), Right (3)
        for i, (dr, dc) in enumerate(directions):
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.size and 0 <= nc < self.size:
                neighbors.append((i, (nr, nc)))
        return neighbors

    def heuristic(self, current, target):
        dist = abs(current[0] - target[0]) + abs(current[1] - target[1])
        return 1.0 / (dist + 0.1)

    def route_single_ant(self, net_idx, start, target, reserved_cells):
        path = [start]
        visited = {start}
        current = start

        while current != target:
            neighbors = self.get_neighbors(current)
            valid_moves = []

            for action, neighbor in neighbors:
                # Single-layer restriction: cannot cross obstacles or previously reserved paths of other traces
                if neighbor not in visited and neighbor not in self.obstacles and neighbor not in reserved_cells:
                    valid_moves.append((action, neighbor))

            if not valid_moves:
                return None

            probs = []
            for action, neighbor in valid_moves:
                tau = self.pheromones[net_idx, current[0], current[1], action] ** self.alpha
                eta = self.heuristic(neighbor, target) ** self.beta
                probs.append(tau * eta)

            sum_probs = sum(probs)
            if sum_probs == 0:
                probs = [1.0 / len(valid_moves)] * len(valid_moves)
            else:
                probs = [p / sum_probs for p in probs]

            chosen_idx = np.random.choice(len(valid_moves), p=probs)
            action, next_pos = valid_moves[chosen_idx]

            path.append(next_pos)
            visited.add(next_pos)
            current = next_pos

            if len(path) > self.size * self.size:
                return None

        return path

    def run_routing(self):
        best_routes = {i: [] for i in range(self.num_nets)}
        best_lengths = {i: float('inf') for i in range(self.num_nets)}

        for gen in range(self.generations):
            reserved_globally = set()

            for net_idx, (start, target) in enumerate(self.nets):
                gen_paths = []

                for ant in range(self.num_ants):
                    path = self.route_single_ant(net_idx, start, target, reserved_globally)
                    if path is not None:
                        gen_paths.append(path)

                if gen_paths:
                    shortest_path = min(gen_paths, key=len)
                    if len(shortest_path) < best_lengths[net_idx]:
                        best_lengths[net_idx] = len(shortest_path)
                        best_routes[net_idx] = shortest_path

                    # Lock the single-layer grid cells for this trace so subsequent traces cannot cross it
                    reserved_globally.update(shortest_path)

                    self.pheromones[net_idx] *= (1.0 - self.evaporation)
                    for path in gen_paths:
                        deposit = self.Q / len(path)
                        for i in range(len(path) - 1):
                            r, c = path[i]
                            nr, nc = path[i+1]
                            dr, dc = nr - r, nc - c
                            action_idx = [(-1,0), (1,0), (0,-1), (0,1)].index((dr, dc))
                            self.pheromones[net_idx, r, c, action_idx] += deposit

        return best_routes

# --- Interactive User Configuration ---

# 1. Grid Size
n = int(input("Enter the size of the board (n x n, e.g., 10): ") or 10)

# 2. Number of Traces
num_traces = int(input("Enter number of traces/nets to route: ") or 2)

# 3. Dynamic Trace Inputs
nets = []
for i in range(num_traces):
    print(f"\n--- Trace {i+1} ---")
    sr = int(input(f"Enter Start Row (0 to {n-1}): ") or 0)
    sc = int(input(f"Enter Start Col (0 to {n-1}): ") or 0)
    er = int(input(f"Enter End Row (0 to {n-1}): ") or (n-1))
    ec = int(input(f"Enter End Col (0 to {n-1}): ") or (n-1))
    nets.append(((sr, sc), (er, ec)))

# 4. Obstacles
obstacles = set()
obs_input = input("\nEnter coordinates of any obstacles (format: r,c; r,c or press Enter for none): ")
if obs_input.strip():
    for pair in obs_input.split(';'):
        if ',' in pair:
            r, c = map(int, pair.strip().split(','))
            obstacles.add((r, c))

print(f"\nInitializing single-layer ACO routing on {n}x{n} grid...")
print(f"Active Traces to route: { {i+1: net for i, net in enumerate(nets)} }")
print(f"Obstacles at: {list(obstacles)}")

# Run the Single-Layer ACO Optimizer
aco = AntColonyPCB(size=n, nets=nets, obstacles=obstacles, num_ants=15, generations=60)
best_routes = aco.run_routing()

# Generate final layout output array
layout_grid = np.zeros((n, n), dtype=int)

for net_idx, path in best_routes.items():
    trace_id = net_idx + 1
    if path:
        for (r, c) in path:
            layout_grid[r, c] = trace_id
    else:
        print(f"Warning: Could not find routing path for Trace {trace_id}")

# Output the grid
print("\n--- Final Output Grid (Trace IDs) ---")
print(layout_grid)


Enter the size of the board (n x n, e.g., 10): 5
Enter number of traces/nets to route: 2

--- Trace 1 ---
Enter Start Row (0 to 4): 0
Enter Start Col (0 to 4): 3
Enter End Row (0 to 4): 4
Enter End Col (0 to 4): 4

--- Trace 2 ---
Enter Start Row (0 to 4): 0
Enter Start Col (0 to 4): 0
Enter End Row (0 to 4): 4
Enter End Col (0 to 4): 0

Enter coordinates of any obstacles (format: r,c; r,c or press Enter for none): 

Initializing single-layer ACO routing on 5x5 grid...
Active Traces to route: {1: ((0, 3), (4, 4)), 2: ((0, 0), (4, 0))}
Obstacles at: []

--- Final Output Grid (Trace IDs) ---
[[2 0 0 1 1]
 [2 0 0 0 1]
 [2 0 0 0 1]
 [2 0 0 0 1]
 [2 0 0 0 1]]
